# Data pre-processing

In [ ]:
import pandas as pd
import nltk
import string
import re

In [ ]:
nltk.download('stopwords')
stopwords=nltk.corpus.stopwords.words('english')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
def preprocess(text):
    # 1: URL handling
    urls  = re.findall('http[s]*://t.co/\w{10}',text)
    for url in urls:
        text = text.replace(url," ")
    # 2: lower case
    text = text.lower()
    # 3: Apostrophe handling
    text = text.replace("'s","").replace("'","")
    # 4: Punctuation and space handling
    translator = str.maketrans(string.punctuation, ' ' * len(string.punctuation))
    text = text.translate(translator)
    # 6: Stopwords
    text = text.replace("rt","")
    text = [w for w in text.split() if not w in stopwords]
    return " ".join(text)

In [ ]:
df = pd.read_csv("newdata.csv", header=None,converters={0: lambda x: " ".join(x.strip("[]").replace("'","").split(", "))})
df

,0,1
0,a neuron doesn t realize it s a neuron,http://pbs.twimg.com/profile_images/1587290337...
1,cookiecrumb35 pretty much or at least one of t...,http://pbs.twimg.com/profile_images/1587290337...
2,with a lot of room for improvement,http://pbs.twimg.com/profile_images/1587290337...
3,because it consists of billion of bidirectiona...,http://pbs.twimg.com/profile_images/1587290337...
4,alizafarsays mrbeast true twitter ha amazing c...,http://pbs.twimg.com/profile_images/1587290337...
...,...,...
9971,this is a government that delivers on what the...,http://pbs.twimg.com/profile_images/1500170386...
9972,we need to come together a a party and focus o...,http://pbs.twimg.com/profile_images/1500170386...
9973,president zelenskyyua just updated me on the o...,http://pbs.twimg.com/profile_images/1500170386...
9974,congratulation to garethbale11 and the rest of...,http://pbs.twimg.com/profile_images/1500170386...


In [ ]:
df = df.rename(columns={0: "tweet", 1: "gender"})
df["tweet"] = df["tweet"].apply(lambda x : preprocess(x))
df["len"] = df["tweet"].apply(lambda x: len(x.split()))
df = df[df["len"] > 3]
df = df.drop("len",axis = 1)
df = df.reset_index()
df = df.drop("index",axis = 1)
df["gender"] = df["gender"].apply(lambda x : x.split("images/")[1])
df

,tweet,gender
0,cookiecrumb35 pretty much least one hive mind,1587290337587904512/Y4s_eu5O_normal.jpg
1,consists billion bidirectional interaction per...,1587290337587904512/Y4s_eu5O_normal.jpg
2,alizafarsays mrbeast true twitter ha amazing c...,1587290337587904512/Y4s_eu5O_normal.jpg
3,davidsacks entitled elite mad pay 8 month mad ...,1587290337587904512/Y4s_eu5O_normal.jpg
4,spacex deployment eutelsat hotbird 13g confirmed,1587290337587904512/Y4s_eu5O_normal.jpg
...,...,...
7940,government delivers people country care focuse...,1500170386520129536/Rr2G6A-N_normal.jpg
7941,need come together pay focus government help p...,1500170386520129536/Rr2G6A-N_normal.jpg
7942,president zelenskyyua updated ongoing battle r...,1500170386520129536/Rr2G6A-N_normal.jpg
7943,congratulation garethbale11 rest cymru squad q...,1500170386520129536/Rr2G6A-N_normal.jpg


# url to image


In [ ]:
import os
def url(x):
  return "http://pbs.twimg.com/profile_images/" + x + "\n"
urls = df["gender"].drop_duplicates().apply(lambda x: url(x)).tolist()
with open("urls.txt","w") as f:
  f.writelines(urls)

In [ ]:
!wget -i urls.txt -P images

In [ ]:
!zip -r images.zip images

In [ ]:
df["gender"] = df["gender"].apply(lambda x : x.split("/")[1])
df_gender = pd.read_csv("file_gender.csv")
df = pd.merge(df,df_gender,left_on = "gender", right_on = "0")
df = df.loc[:,["tweet","1"]]
df = df.reset_index().drop("index",axis=1)
df = df.rename(columns={"1":"gender"})

In [ ]:
df.to_csv("tweet_gender.csv")

#Gender classification

In [ ]:
!pip install tensorflow
!pip install keras
from keras_preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, Softmax, Dropout, Activation
from keras.layers import SimpleRNN, LSTM, Embedding, Bidirectional
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.callbacks import ModelCheckpoint
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

--2022-11-03 09:28:57--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2022-11-03 09:28:58--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2022-11-03 09:28:58--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [ ]:
df = pd.read_csv("tweet_gender.csv")
df_test = pd.concat([df[df["gender"] == "Male"][:250],df[df["gender"] == "Female"][:250]],ignore_index = False)
df_train = df.drop(df_test.index)

x_train = df_train["tweet"]
y_train = df_train["gender"]
x_test = df_test["tweet"]
y_test = df_test["gender"]

encoder = Tokenizer(lower=False) 
encoder.fit_on_texts(x_train) 
x_train = encoder.texts_to_sequences(x_train) 
x_test = encoder.texts_to_sequences(x_test)
total_words = len(encoder.word_index) + 1

def get_max_length():
    review_length = []
    for review in x_train:
        review_length.append(len(review))
    return int(np.ceil(np.mean(review_length)))
MAX_SEQUENCE_LENGTH=get_max_length()

from keras_preprocessing.sequence import pad_sequences
x_train = pad_sequences(x_train, maxlen=MAX_SEQUENCE_LENGTH, value=0, padding='post')
x_test = pad_sequences(x_test, maxlen=MAX_SEQUENCE_LENGTH, value=0, padding='post')

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train=le.fit_transform(y_train)
y_test=le.transform(y_test)

In [ ]:
embeddings_index = {}
with open("glove.6B.300d.txt") as f:
    for line in f:
        word, coefs = line.split(maxsplit=1)
        coefs = np.fromstring(coefs, "f", sep=" ")
        embeddings_index[word] = coefs
word_index = encoder.word_index
embedding_size=300
embedding_matrix = np.zeros((total_words, embedding_size))
for word, i in word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

In [ ]:
import tensorflow
from keras.initializers import Constant
model=Sequential()
model.add(Embedding(total_words,300,embeddings_initializer=Constant(embedding_matrix),input_length=MAX_SEQUENCE_LENGTH,trainable=True))
model.add(LSTM(768, dropout = 0.5))
model.add(Dense(1,activation='sigmoid'))  
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
checkpoint = ModelCheckpoint(">66.hdf5", monitor='val_accuracy', verbose=1,save_best_only=True, mode='auto',save_weights_only=False)
callback = tensorflow.keras.callbacks.EarlyStopping(monitor='loss', patience=3)
history= model.fit(x_train, y_train, epochs=100,callbacks=[checkpoint,callback],validation_data=(x_test, y_test))

Epoch 1/100
230/233 [============================>.] - ETA: 0s - loss: 0.6586 - accuracy: 0.6030
Epoch 1: val_accuracy improved from -inf to 0.61800, saving model to >66.hdf5
233/233 [==============================] - 11s 40ms/step - loss: 0.6586 - accuracy: 0.6038 - val_loss: 0.6835 - val_accuracy: 0.6180
Epoch 2/100
231/233 [============================>.] - ETA: 0s - loss: 0.4791 - accuracy: 0.7753
Epoch 2: val_accuracy improved from 0.61800 to 0.66000, saving model to >66.hdf5
233/233 [==============================] - 6s 26ms/step - loss: 0.4788 - accuracy: 0.7756 - val_loss: 0.5819 - val_accuracy: 0.6600
Epoch 3/100
232/233 [============================>.] - ETA: 0s - loss: 0.3100 - accuracy: 0.8738
Epoch 3: val_accuracy improved from 0.66000 to 0.70400, saving model to >66.hdf5
233/233 [==============================] - 6s 26ms/step - loss: 0.3099 - accuracy: 0.8739 - val_loss: 0.5863 - val_accuracy: 0.7040
Epoch 4/100
233/233 [==============================] - ETA: 0s - loss: 0